# Week 1 Project: Build a Coding Agent That Fixes Failing Tests

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/new-programs-ik/Applied-Agentic-AI-for-SWEs/blob/main/Week%201/project/Fix_Failing_Tests_Agent.ipynb)

## The problem

Every engineering team knows the routine: CI goes red, someone opens the test output, reads the error, opens a few files, finds the real cause (often in a different file from the failing test), makes a fix, runs the tests again, and repeats until everything is green.

Tools like GitHub Copilot agent mode, Claude Code and Cursor now do this loop on their own. In this project you will build a small version of one **from scratch**, with nothing but an LLM API and simple Python.

## What you will build

An agent that is given a small Python project with failing tests and one goal: **make all the tests pass.** It has four tools:

| Tool | What it does |
|---|---|
| `list_files()` | See what files are in the project |
| `read_file(path)` | Read a file |
| `write_file(path, content)` | Save a fixed version of a file |
| `run_tests()` | Run the tests and see what passes and fails |

Nobody tells the agent which files to open, which bug to fix first, or how many attempts it gets. It decides, acts, looks at the test results, and decides again: the **Think → Act → Observe** loop from class.

## What you will learn

1. How a real agent loop works, end to end, with no framework hiding it.
2. The difference between a **workflow** (you decide the steps) and an **agent** (the model decides the steps), by building both on the same problem.
3. Why tools that give **real feedback** (here: test results) are the best defence against hallucination.
4. Basic guardrails every agent needs: limited file access, protected files, a step limit, and checking the result yourself.

## Why this is the Week 1 project

This project uses **only** what this week's class covers, and practises every part of it:

| Concept from this week's class | Where it shows up in this notebook |
|---|---|
| Calling LLMs programmatically | One OpenAI-SDK client, routed through OpenRouter, exactly as in the "Calling LLMs" notebook |
| Prompt engineering: context, structure, constraints | The agent's system prompt uses the `## Role / ## Task / ## Rules / ## Output` structure from the prompt engineering slide |
| Limitations of LLMs: no actions, hallucinations | The model can't run code or see your files on its own, and it can be confidently wrong. Tools give it actions; `run_tests` catches wrong fixes |
| Agent = (Prompt + Tools + Memory) × LLM | Prompt = system prompt, Tools = the four functions, Memory = the list of messages that carries every earlier attempt |
| Agents and tools: retrieve data, execute actions, run code | `read_file` retrieves, `write_file` acts, `run_tests` runs code |
| Workflows vs. autonomous agents | Part 1 builds a fixed workflow; Part 2 builds the agent. You compare them on the same bugs |
| The agent loop / ReAct (Think → Act → Observe) | The `run_agent` loop prints each step so you can follow what the agent is doing |
| Planning and reflection | When a fix fails, the agent has to read the new error and change its fix |

It is also sized for Week 1:

- **Beginner-friendly Python.** The code uses only functions, `if`, `for` loops, lists, dictionaries and strings. Setup code that isn't about agents is tucked into one helper cell you can skip. If Python is new to you, read the short primer below first.
- **No frameworks.** No LangChain, LangGraph, vector databases or deployment. Those come in later weeks, and they will make more sense once you have written the loop yourself.
- **Runs in Colab with nothing to install.** The only secret you need is `OPENROUTER_APIKEY`.
- **Cheap.** A full run with the default model costs about one cent.
- **Real, not simulated.** The tools change real files and run real tests. If the agent's fix is wrong, the tests say so, and the agent has to deal with it.

## Python primer: everything you need for this notebook

You already know how to program. This table maps the Python used in this notebook to Java and JavaScript.

| Python | Meaning | Java / JavaScript |
|---|---|---|
| `def add(a, b):` and an indented body | Define a function. **Indentation replaces `{ }`** | `int add(int a, int b) { ... }` / `function add(a, b) { ... }` |
| `# a comment` | Comment | `// a comment` |
| `name = "Ana"` | Variables need no type or `let`/`var` | `String name = "Ana";` / `let name = "Ana";` |
| `f"Hello {name}"` | String with values inserted | `"Hello " + name` / `` `Hello ${name}` `` |
| `items = ["a", "b"]`, `items.append("c")` | A list | `ArrayList` / array with `.push()` |
| `user = {"name": "Ana", "age": 30}`, `user["name"]` | A dictionary (key → value) | `HashMap` / object |
| `for item in items:` | Loop over a list | `for (String item : items)` / `for (const item of items)` |
| `for step in range(1, 21):` | Count from 1 to 20 | `for (int step = 1; step <= 20; step++)` |
| `if x: ... elif y: ... else: ...` | `elif` means "else if" | `if / else if / else` |
| `True`, `False`, `None` | Booleans and null (note the capital letters) | `true`, `false`, `null` |
| `text[:100]` and `text[-100:]` | First 100 characters, last 100 characters | `text.substring(0, 100)` / `text.slice(-100)` |
| `passed, output = run_pytest()` | A function can return two values at once | Returning an object with two fields |
| `try: ... except Exception as error: ...` | Catch errors | `try { ... } catch (Exception error) { ... }` |
| `with open(path) as f: content = f.read()` | Open a file and close it automatically | try-with-resources / `fs.readFileSync` |
| `json.loads(text)` / `json.dumps(data)` | Parse JSON / turn data into JSON | `JSON.parse` / `JSON.stringify` |

Python also writes logic operators as words: `not`, `and`, `or`.

# Setup

In [ ]:
%pip install -q openai pytest

In [ ]:
import os
import json
from openai import OpenAI

### Connect to the model

As in the other Week 1 notebooks, we use the official `openai` SDK pointed at OpenRouter. Add `OPENROUTER_APIKEY` in Colab Secrets (the key icon in the left sidebar) and allow this notebook to read it.

In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get("OPENROUTER_APIKEY")
except ImportError:
    # Not running in Colab: read the key from an environment variable instead
    api_key = os.environ.get("OPENROUTER_APIKEY")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

# Any OpenRouter model that supports tool calling works here, for example
# "anthropic/claude-haiku-4.5", "google/gemini-2.5-flash" or "openai/gpt-4o".
MODEL = "openai/gpt-4.1-mini"

### Helper code

The next cell contains setup code that isn't about agents: the files of the buggy project, a function that runs the tests, and a function that shows what changed. **Just run it.** In Colab it is collapsed; you can double-click it later if you are curious.

It gives you these functions:

| Function | What it does |
|---|---|
| `create_repo()` | Creates (or resets) the buggy project in a folder called `shopkit_repo` |
| `run_pytest()` | Runs the tests. Returns two values: `True`/`False` for "all passed", and the test output as text |
| `show_diff()` | Shows every line that changed since the project was created, like a pull request |
| `extract_code(text)` | Pulls the Python code out of a model reply that wraps it in a ```` ```python ```` block |
| `get_cost(response)` | Returns the cost in dollars of one model call, as reported by OpenRouter |

In [ ]:
#@title Helper code (run this cell, no need to read it)
import difflib
import re
import subprocess
import sys

REPO_DIR = os.path.abspath("shopkit_repo")

ORIGINAL_FILES = {
    "shopkit/__init__.py": "",

    "shopkit/pagination.py": '''"""Split long product lists into pages for display."""
import math


def paginate(items, page, page_size):
    """Return the items on the given page. Pages are numbered from 1."""
    start = page * page_size
    return items[start:start + page_size]


def total_pages(item_count, page_size):
    """Return how many pages are needed to show item_count items."""
    return item_count // page_size
''',

    "shopkit/pricing.py": '''"""Price calculations shared by the cart and checkout."""


def apply_discount(price, percent):
    """Return the price after a percentage discount.

    Example: apply_discount(200, 10) -> 180.0
    """
    return round(price * percent / 100, 2)
''',

    "shopkit/cart.py": '''"""A minimal shopping cart."""
from shopkit.pricing import apply_discount


class Cart:
    def __init__(self):
        self.items = []

    def add_item(self, name, unit_price, quantity=1):
        self.items.append({"name": name, "unit_price": unit_price, "quantity": quantity})

    def subtotal(self):
        return round(sum(i["unit_price"] * i["quantity"] for i in self.items), 2)

    def total(self, discount_percent=0):
        """Subtotal after an optional whole-cart percentage discount."""
        return apply_discount(self.subtotal(), discount_percent)
''',

    "shopkit/text.py": '''"""Text helpers for product pages."""


def slugify(name):
    """Turn a product name into a URL slug.

    Lowercase ASCII letters and digits only, words joined by single hyphens,
    accents removed, apostrophes dropped, no leading or trailing hyphens.
    """
    return name.lower().replace(" ", "-")
''',

    "tests/test_pagination.py": '''from shopkit.pagination import paginate, total_pages

ITEMS = list(range(1, 26))  # 25 products


def test_first_page():
    assert paginate(ITEMS, page=1, page_size=10) == list(range(1, 11))


def test_last_partial_page():
    assert paginate(ITEMS, page=3, page_size=10) == [21, 22, 23, 24, 25]


def test_total_pages_rounds_up():
    assert total_pages(25, 10) == 3


def test_total_pages_exact():
    assert total_pages(20, 10) == 2
''',

    "tests/test_cart.py": '''from shopkit.cart import Cart


def make_cart():
    cart = Cart()
    cart.add_item("T-shirt", 20.00, quantity=2)
    cart.add_item("Cap", 15.50)
    return cart


def test_subtotal():
    assert make_cart().subtotal() == 55.50


def test_total_without_discount():
    assert make_cart().total() == 55.50


def test_total_with_discount():
    assert make_cart().total(discount_percent=10) == 49.95
''',

    "tests/test_text.py": '''from shopkit.text import slugify


def test_simple_name():
    assert slugify("Blue Shirt") == "blue-shirt"


def test_extra_whitespace():
    assert slugify("  Red   Hat  ") == "red-hat"


def test_punctuation_and_apostrophes():
    assert slugify("Men's T-Shirt (XL)") == "mens-t-shirt-xl"


def test_accents_are_removed():
    assert slugify("Café Crème Mug") == "cafe-creme-mug"
''',
}


def create_repo():
    for path, content in ORIGINAL_FILES.items():
        full_path = os.path.join(REPO_DIR, path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"Project ready at {REPO_DIR}")


def run_pytest():
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--tb=short", "-p", "no:cacheprovider"],
        cwd=REPO_DIR, capture_output=True, text=True, timeout=60,
    )
    return result.returncode == 0, result.stdout + result.stderr


def show_diff():
    for path, original in ORIGINAL_FILES.items():
        with open(os.path.join(REPO_DIR, path), encoding="utf-8") as f:
            current = f.read()
        if current != original:
            diff = difflib.unified_diff(
                original.splitlines(keepends=True), current.splitlines(keepends=True),
                fromfile=f"a/{path}", tofile=f"b/{path}",
            )
            print("".join(diff))


def extract_code(text):
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.S)
    if match:
        return match.group(1)
    return text


def get_cost(response):
    cost = (response.usage.model_extra or {}).get("cost")
    if cost is None:
        return 0
    return cost


print("Helpers loaded.")

# Step 1: Create the buggy project

`create_repo()` writes a tiny e-commerce library called `shopkit` into the `shopkit_repo` folder:

```
shopkit_repo/
├── shopkit/
│   ├── pagination.py   # split product lists into pages
│   ├── pricing.py      # discount maths
│   ├── cart.py         # shopping cart totals (uses pricing.py)
│   └── text.py         # turn product names into web-address "slugs"
└── tests/
    ├── test_pagination.py
    ├── test_cart.py
    └── test_text.py
```

Some bugs have been planted. Some are easy to spot. At least one is in a different file from the test that fails, and at least one has an obvious fix that is only partly right. The answer key is at the end of the notebook. Try not to read it before the agent runs.

You can look at the files in Colab's file browser (the folder icon on the left). Run `create_repo()` again at any time to reset the project to its buggy starting state.

In [ ]:
create_repo()

### See the failures yourself

Before handing anything to an LLM, run the tests the way a developer would. This output is exactly what the agent will see when it calls `run_tests`.

Each `F` is a failing test and each `.` is a passing one. Below that, each failure shows the expected value and the actual value.

In [ ]:
passed, output = run_pytest()
print(output)

# Step 2: Build the tools

Tools are ordinary Python functions. Three design rules make them safe and useful for an agent:

1. **Return a message, don't crash.** If the agent asks for a file that doesn't exist, it should get a clear error message it can react to. Errors are useful observations too.
2. **Stay inside the project folder.** The agent must not be able to read or write files elsewhere on the machine, such as `../../secrets.txt`.
3. **Protect what defines "done".** The agent may not edit anything in `tests/`. Otherwise the easiest way to make tests pass is to change the tests, and an agent chasing a goal will sometimes find that shortcut.

We also cut long outputs short (`MAX_OUTPUT`), because everything a tool returns is sent back to the model on every later step, and that costs tokens.

In [ ]:
MAX_OUTPUT = 4000  # the most characters a tool sends back to the model


def is_inside_repo(path):
    """Guardrail: True only if the path points inside the project folder."""
    full_path = os.path.abspath(os.path.join(REPO_DIR, path))
    return full_path.startswith(REPO_DIR + os.sep)


def list_files():
    files = []
    for folder, subfolders, filenames in os.walk(REPO_DIR):
        for filename in filenames:
            if filename.endswith(".py"):
                full_path = os.path.join(folder, filename)
                relative_path = os.path.relpath(full_path, REPO_DIR).replace("\\", "/")
                files.append(relative_path)
    files.sort()
    return "\n".join(files)  # one file per line


def read_file(path):
    if not is_inside_repo(path):
        return f"ERROR: {path} is outside the project folder"

    full_path = os.path.join(REPO_DIR, path)
    if not os.path.isfile(full_path):
        return f"ERROR: file not found: {path}"

    with open(full_path, encoding="utf-8") as f:
        content = f.read()
    return content[:MAX_OUTPUT]


def write_file(path, content):
    if not is_inside_repo(path):
        return f"ERROR: {path} is outside the project folder"

    clean_path = os.path.normpath(path).replace("\\", "/")  # e.g. "./tests/a.py" becomes "tests/a.py"
    if clean_path.startswith("tests/"):
        return "ERROR: files in tests/ are read-only. Fix the source code, not the tests."
    if not clean_path.endswith(".py"):
        return "ERROR: only .py files can be written."

    full_path = os.path.join(REPO_DIR, clean_path)
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Wrote {len(content)} characters to {path}"


def run_tests():
    passed, output = run_pytest()
    if passed:
        status = "ALL TESTS PASSED"
    else:
        status = "SOME TESTS FAILED"
    return status + "\n\n" + output[-MAX_OUTPUT:]

Try the guardrails before the agent does:

In [ ]:
print(read_file("../secrets.txt"))
print(write_file("tests/test_cart.py", "def test_ok(): pass"))
print(read_file("shopkit/missing.py"))

### Connect the tool names to the functions

When the model wants to use a tool, it sends us the tool's **name** and its **arguments** as a JSON string, for example:

```
name:      "read_file"
arguments: '{"path": "shopkit/cart.py"}'
```

`call_tool` turns that request into a real function call. If anything goes wrong (bad JSON, a missing argument, an unknown tool), it returns an error message instead of crashing, so the agent can see the problem and try again.

In [ ]:
def call_tool(name, arguments_json):
    try:
        args = json.loads(arguments_json)  # JSON text -> Python dictionary

        if name == "list_files":
            return list_files()
        elif name == "read_file":
            return read_file(args["path"])
        elif name == "write_file":
            return write_file(args["path"], args["content"])
        elif name == "run_tests":
            return run_tests()
        else:
            return f"ERROR: unknown tool {name}"

    except KeyError as error:
        return f"ERROR: {name} is missing the argument {error}"
    except Exception as error:
        return f"ERROR: could not run {name}: {error}"


print(call_tool("read_file", '{"path": "shopkit/pricing.py"}'))

### Describe the tools to the model

The model never sees our Python functions, only these descriptions (written in a format called JSON Schema). The descriptions matter: they are how the model decides **when** to use each tool. Notice that `write_file` says it replaces the whole file, so the model knows to send the complete file and not just the changed lines.

In [ ]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all Python files in the project, as paths relative to the project folder.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the full contents of a file in the project.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path inside the project, e.g. 'shopkit/cart.py'"},
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": (
                "Replace the ENTIRE contents of a source file with new content. "
                "Always send the complete file, not just the changed lines. Files in tests/ are read-only."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path inside the project, e.g. 'shopkit/cart.py'"},
                    "content": {"type": "string", "description": "The complete new contents of the file."},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run all the tests and return a pass/fail summary with details of any failures.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
]

# Part 1: The workflow version (you decide the steps)

Before building the agent, let's solve the problem the "traditional" way: a **fixed workflow** where we decide every step.

1. Run the tests once.
2. For each failing test file, assume the bug is in the source file with the matching name (`tests/test_cart.py` → `shopkit/cart.py`). We write this rule down in `TEST_TO_SOURCE`.
3. Send that test file, that source file and the test output to the LLM in **one** call, and ask for the fixed file.
4. Save it. Run the tests one last time and report.

This is a perfectly reasonable design, and for many well-defined jobs a workflow like this is the right choice: it is predictable, and it makes a known, fixed number of LLM calls. But look at the assumptions we built in: the bug is always in the file with the matching name, and one attempt is enough. Watch what happens when the real problem doesn't follow them.

In [ ]:
# Our hard-coded rule: which source file each test file checks
TEST_TO_SOURCE = {
    "tests/test_pagination.py": "shopkit/pagination.py",
    "tests/test_cart.py": "shopkit/cart.py",
    "tests/test_text.py": "shopkit/text.py",
}


def run_workflow():
    passed, output = run_pytest()
    total_cost = 0

    for test_file, source_file in TEST_TO_SOURCE.items():
        if test_file not in output:
            continue  # no failures in this test file, skip it

        prompt = (
            "These tests are failing.\n\n"
            f"## Test file: {test_file}\n{read_file(test_file)}\n\n"
            f"## Source file: {source_file}\n{read_file(source_file)}\n\n"
            f"## Test output\n{output}\n\n"
            f"Return the complete fixed version of {source_file} in a single ```python code block."
        )
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        total_cost = total_cost + get_cost(response)

        new_code = extract_code(response.choices[0].message.content)
        print("LLM call ->", write_file(source_file, new_code))

    passed, output = run_pytest()
    print(f"\nWorkflow cost: ${total_cost:.4f}")
    if passed:
        print("Result: ALL TESTS PASSED")
    else:
        print("Result: SOME TESTS FAILED")
    print(output)


create_repo()
run_workflow()

### What usually happens

Results vary between runs and models, but in our test runs the workflow **always left the two cart tests failing**. The real bug is in `pricing.py`, but our rule only ever showed the model `cart.py`. Sometimes a model even "fixes" it with a workaround inside `cart.py`, which turns the tests green but leaves the real bug in place for everyone else who uses `pricing.py`.

Either way, the weakness is the same: **we decided the steps in advance, and the problem didn't fit them.** The workflow also has no step for "look at the new error and try again", because we never wrote one.

Look at what the workflow changed:

In [ ]:
show_diff()

# Part 2: The agent version (the model decides the steps)

Now we give the model the goal, the tools and a loop, and let it decide everything else.

### The system prompt

The prompt follows the structure from the prompt engineering slide: **Role**, **Task**, **Rules** (constraints) and **Output**. Compare it with the workflow above: it says *what* "done" looks like and *what is not allowed*, but not *which files to open in which order*. That freedom is what makes this an agent.

In [ ]:
SYSTEM_PROMPT = """## Role
You are a careful senior Python engineer fixing a failing test suite.

## Task
Make every test in the project pass by fixing bugs in the source code.

## Rules
- Run the tests first to see what is failing.
- Read the relevant source files before changing anything. A failing test can be caused by a bug in a different file from the one it tests: follow the imports.
- Fix the root cause, where the bug actually lives. Do not work around a bug in one file by changing another.
- Never edit files in tests/. The tests define the correct behaviour.
- Keep changes small. Do not rewrite code that already works.
- After every change, run the tests again. If something still fails, read the new error and change your fix.
- Stop when all tests pass.

## Output
When you are finished, reply with a short summary: for each bug, the file, the root cause, and the fix."""

### The agent loop

This is the whole agent. Each pass through the loop is one **Think → Act → Observe** cycle:

- **Think:** send all the messages so far to the model. It either asks to use some tools, or gives its final answer.
- **Act:** run each tool the model asked for, using `call_tool`.
- **Observe:** add each tool's result to the messages, so the model sees it next time.

Two details are worth noticing:

- **`max_steps`** is a hard stop. Agents can get stuck repeating themselves, and every step costs money. Real agents always have a limit like this.
- **Memory is just the `messages` list.** The model remembers nothing between calls. It "remembers" its earlier failed fix only because we send the whole conversation every time. That is also why each step costs a little more than the one before.

In [ ]:
def run_agent(goal, max_steps=20):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": goal},
    ]
    total_cost = 0
    finished = False

    for step in range(1, max_steps + 1):
        # THINK: the model reads all the messages so far and decides what to do next
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        total_cost = total_cost + get_cost(response)
        message = response.choices[0].message
        messages.append(message)  # remember what the model said

        if message.content:
            print(f"\n[Step {step}] THINK: {message.content}")

        # No tool requests means the model thinks it is done
        if not message.tool_calls:
            finished = True
            break

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            arguments = tool_call.function.arguments  # a JSON string

            # ACT: run the tool the model asked for
            print(f"[Step {step}] ACT: {name} {arguments[:60]}")
            result = call_tool(name, arguments)

            # OBSERVE: add the result to the messages so the model sees it next step
            first_line = result.split("\n")[0]
            print(f"[Step {step}] OBSERVE: {first_line}")
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    if not finished:
        print(f"\nStopped: reached max_steps={max_steps} before the agent finished.")
    print(f"\n--- {step} model calls, cost ${total_cost:.4f} ---")
    return messages

### Run it

We reset the project first, so the agent starts from the same buggy code the workflow did.

In [ ]:
create_repo()
history = run_agent("The tests in this project are failing. Please fix them.")

# Step 3: Don't trust the agent. Check its work.

The agent *says* it is done. An engineer would never merge a pull request just because the author says it works, so we check for ourselves:

1. Run the tests ourselves, separately from the agent.
2. Review the diff, the same way you would review a teammate's pull request. Did it fix the root cause? Did it touch anything it shouldn't have?

In [ ]:
passed, output = run_pytest()
if passed:
    print("CHECKED: all tests pass")
else:
    print("NOT DONE: tests still fail")
print(output)
show_diff()

### Reading the agent's steps

Scroll back up through the agent's output and look for these moments. They are the whole point of the agent loop:

- **Exploring:** the agent reads files nobody told it about, for example opening `pricing.py` because `cart.py` uses it.
- **Acting on evidence:** each `write_file` comes after reading the code and seeing a test failure, not from a guess.
- **Reflection:** a `run_tests` result shows a fix that only partly worked, and the next step changes that fix. (Strong models sometimes get every fix right the first time. Run it again, or try a smaller model, to see this happen.)
- **Stopping:** the agent stops asking for tools once `run_tests` reports success.

Then compare with Part 1:

| | Workflow (Part 1) | Agent (Part 2) |
|---|---|---|
| Who decides the next step? | Our code | The model |
| Number of LLM calls | Fixed: one per failing test file | Unknown in advance: as many as it needs (up to `max_steps`) |
| Finds a bug in an unexpected file | Only if we predicted it | Yes, by following imports |
| Recovers from a fix that didn't work | No | Yes: it reads the new error and tries again |
| Predictability and cost | High / low | Lower / higher |

Neither is "better" in general. As the class slide says, most real systems are **hybrids**: a fixed workflow for the overall stages, with an agent given freedom inside one stage. A CI bot might always run "reproduce → agent fixes → tests check → human reviews", while the "agent fixes" stage is open-ended.

# Exercises

Pick at least two. They are listed from easiest to hardest.

1. **Compare models.** Change `MODEL` to two other models and run the agent again. Write down the number of steps, the cost, and whether the fix was correct. Which would you pick for a CI bot, and why?
2. **Try to break the guardrails.** Change the goal to "Make the tests pass by any means necessary." Does the agent try to edit `tests/`? What does it do when the tool refuses?
3. **Plan first.** Add a rule to the system prompt asking the agent to write a numbered plan before its first tool call. Does planning change the number of steps?
4. **Add a bug of your own.** Add a new bug and a test that catches it to `ORIGINAL_FILES` in the helper cell, then run the agent again. Can you write a bug the agent can't fix in 20 steps?
5. **Add a `search_code(text)` tool** that returns every line containing `text`, with its file name and line number. You need to write the function, add it to `call_tool`, and describe it in `TOOLS_SCHEMA`. Does the agent use it instead of reading whole files?
6. **Smarter edits.** Add a `replace_in_file(path, old, new)` tool that changes only one exact piece of text, using Python's `content.replace(old, new)`. Why do real coding agents prefer this over rewriting the whole file? (Hint: think about tokens and accidental damage to the rest of the file.)

# Ship it to your portfolio

As with every agent in this course: put the code in a small Python project, write a README (the problem, how the agent works, its tools and guardrails, a sample run, limitations, next steps), and publish it on your GitHub.

---

# Answer key (instructors, or after you have run the agent)

| # | Bug | Failing tests | Root cause | Fix | What it tests in the agent |
|---|---|---|---|---|---|
| 1 | `paginate` uses `page * page_size`, but pages start at 1 | `test_first_page`, `test_last_partial_page` | Off-by-one in `pagination.py` | `start = (page - 1) * page_size` | Basic read → fix → check |
| 2 | `total_pages` rounds down | `test_total_pages_rounds_up` | `//` (whole-number division) in `pagination.py` | `math.ceil(item_count / page_size)` (`math` is already imported) | A second bug in the same file |
| 3 | `apply_discount` returns the discount amount, not the discounted price | All of `test_cart.py` except `test_subtotal` | The bug is in `pricing.py`, but only `test_cart.py` fails | `round(price * (100 - percent) / 100, 2)` | Following imports to the real cause, and not working around it in `cart.py` |
| 4 | `slugify` only replaces single spaces | 3 tests in `test_text.py` | Too-simple code in `text.py` | Remove accents with `unicodedata`, drop apostrophes, replace other characters with `-`, trim `-` from the ends | Reflection: the obvious fix `re.sub(r"[^a-z0-9]+", "-", ...)` still fails the apostrophe and accent tests |

A reference fix for bug 4:

```python
import re
import unicodedata


def slugify(name):
    ascii_name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    ascii_name = ascii_name.lower().replace("'", "")
    return re.sub(r"[^a-z0-9]+", "-", ascii_name).strip("-")
```